In [1]:
import pandas as pd
import onnxruntime as ort
import onnx
import numpy as np
import dice_ml
from dice_ml import Dice

In [2]:
# load training and testing data
training_data = pd.read_csv("for_testing/training_data_2021-11-23 00:00:00.csv")
testing_data = pd.read_csv("for_testing/testing_data_2021-11-23 00:00:00.csv")
print(training_data.loc[43424])

past_profitability_21d       0.025818
past_profitability_63d       0.046051
past_profitability_126d      0.131742
volatility_21d               0.078286
volatility_63d               0.079081
volatility_126d              0.078615
avg_price_21d                8.400078
avg_price_63d                8.368175
avg_price_126d               8.056539
sharpe_21d                   0.329959
sharpe_63d                   0.582737
sharpe_126d                  1.676071
m_21d                         0.21452
m_63d                         0.37516
m_126d                        0.99222
roc_21d                      0.025818
roc_63d                      0.046051
roc_126d                     0.131742
MACD                          0.05093
rsi_14                      63.016488
dco_22                      -0.053505
min_21d                        8.2331
min_63d                      8.146272
min_126d                       7.5163
max_21d                        8.5652
max_63d                        8.5652
max_126d    

In [3]:
training_data = training_data.drop(columns=["col_timestamp"])
testing_data = testing_data.drop(columns=["col_timestamp"])

In [4]:
# load trained model
session = ort.InferenceSession("for_testing/profitability_recommendation_2021-11-23 00:00:00.onnx")

model = onnx.load("for_testing/profitability_recommendation_2021-11-23 00:00:00.onnx")
onnx.checker.check_model(model)

In [5]:
input_name = session.get_inputs()[0].name
label_name = session.get_outputs()[0].name
input_name, label_name

('float_input', 'variable')

In [6]:
query_to_predict = testing_data.drop(columns="target")[0:1].values.astype(np.float32)
print(query_to_predict)

[[-7.38290325e-02  2.13770211e-01  2.58344769e-01  3.10262740e-01
   4.33814436e-01  4.05843347e-01  4.95974493e+00  4.71617842e+00
   4.14708900e+00 -2.36372441e-01  4.93708938e-01  6.37239993e-01
  -3.75000000e-01  8.24999988e-01  9.59999979e-01 -7.38290325e-02
   2.13770211e-01  2.58344769e-01 -3.37277204e-02  3.66351509e+01
   1.23516366e-01  4.61399984e+00  3.91300011e+00  3.31999993e+00
   5.19000006e+00  5.19000006e+00  5.19000006e+00  4.86837721e+00
   4.64618778e+00  4.23288155e+00]]


In [212]:
pred_onx = session.run(None, {input_name: query_to_predict})[0]

In [213]:
pred_onx

array([[0.00463372]], dtype=float32)

## Dice

Works on regression, but it takes too long to generate each counterfactual. CARLA is optimized but only works on classification.

In [7]:


y = training_data["target"]

X = training_data.drop("target", axis=1)

continuous_cols = X.select_dtypes(include=["number"]).columns.tolist()

dice_data = dice_ml.Data(
    dataframe=training_data,
    continuous_features=list(continuous_cols),
    outcome_name="target"
)



In [8]:
y.min(), y.max()

(np.float64(-0.9272918124562632), np.float64(11.238508633824596))

In [9]:
class ONNXWrapper:
    debug_calls = 0
    def __init__(self, session, feature_names):
        self.session = session
        self.input_name = session.get_inputs()[0].name
        self.label_name = session.get_outputs()[0].name
        self.feature_names = feature_names

    def predict(self, X):
        if isinstance(X, pd.DataFrame):
            X = X[self.feature_names].astype(np.float32).values
        else: 
            raise TypeError(f"Expected X to be a pandas DataFrame, but got {type(X).__name__}")

        # Run ONNX inference
        pred = self.session.run([self.label_name], {self.input_name: X})[0]

        # flatten ONNX output so DiCE returns scalars, not lists
        pred = pred.reshape(-1)  

        return pred


In [10]:
onnx_model = ONNXWrapper(session, X.columns)

In [11]:
# One instance
query_instance = testing_data.drop(columns="target")[0:1].astype(np.float32)
print("Factual instance:")
print(query_instance)

Factual instance:
   past_profitability_21d  past_profitability_63d  past_profitability_126d  \
0               -0.073829                 0.21377                 0.258345   

   volatility_21d  volatility_63d  volatility_126d  avg_price_21d  \
0        0.310263        0.433814         0.405843       4.959745   

   avg_price_63d  avg_price_126d  sharpe_21d  ...    dco_22  min_21d  min_63d  \
0       4.716178        4.147089   -0.236372  ...  0.123516    4.614    3.913   

   min_126d  max_21d  max_63d  max_126d  exp_mean_21d  exp_mean_63d  \
0      3.32     5.19     5.19      5.19      4.868377      4.646188   

   exp_mean_126d  
0       4.232882  

[1 rows x 30 columns]


In [ ]:
# Sklearn wrapping
dice_model = dice_ml.Model(model=onnx_model, backend="sklearn", model_type="regressor")

# Generate counterfactual with DiCE
exp = Dice(dice_data, dice_model, method="genetic")  # "random" ou "genetic" ou "kd"

# Generate 2 counterfactuals
dice_exp = exp.generate_counterfactuals(query_instance, total_CFs=2, desired_range=[float(y.min()), float(y.max())])

# Visualisation
dice_exp.visualize_as_dataframe(show_only_changes=True)


100%|██████████| 1/1 [00:00<00:00,  1.31it/s]

Query instance (original outcome : 0.00463371817022562)


,past_profitability_21d,past_profitability_63d,past_profitability_126d,volatility_21d,volatility_63d,volatility_126d,avg_price_21d,avg_price_63d,avg_price_126d,sharpe_21d,...,min_21d,min_63d,min_126d,max_21d,max_63d,max_126d,exp_mean_21d,exp_mean_63d,exp_mean_126d,target
0,-0.073829,0.21377,0.258345,0.310263,0.433814,0.405843,4.959745,4.716178,4.147089,-0.236372,...,4.614,3.913,3.32,5.19,5.19,5.19,4.868377,4.646188,4.232882,0.004634



Diverse Counterfactual set (new outcome: [-0.9272918124562632, 11.238508633824596])


,past_profitability_21d,past_profitability_63d,past_profitability_126d,volatility_21d,volatility_63d,volatility_126d,avg_price_21d,avg_price_63d,avg_price_126d,sharpe_21d,...,min_21d,min_63d,min_126d,max_21d,max_63d,max_126d,exp_mean_21d,exp_mean_63d,exp_mean_126d,target
0,-0.10000000149011612,0.20000000298023224,0.20000000298023224,0.30000001192092896,0.4000000059604645,0.4000000059604645,5.0,4.699999809265137,4.099999904632568,-0.20000000298023224,...,4.599999904632568,3.9000000953674316,3.299999952316284,5.199999809265137,5.199999809265137,5.199999809265137,4.900000095367432,4.639999866485596,4.230000019073486,-0.04089638218283653
0,0.0,0.30000001192092896,0.20000000298023224,0.4000000059604645,0.4000000059604645,0.4000000059604645,5.0,4.599999904632568,4.099999904632568,0.10000000149011612,...,4.599999904632568,3.700000047683716,3.299999952316284,5.199999809265137,5.199999809265137,5.199999809265137,5.0,4.630000114440918,4.190000057220459,-0.06193215027451515


In [13]:
cfs = dice_exp.cf_examples_list[0].final_cfs_df

In [14]:
cfs

,past_profitability_21d,past_profitability_63d,past_profitability_126d,volatility_21d,volatility_63d,volatility_126d,avg_price_21d,avg_price_63d,avg_price_126d,sharpe_21d,...,min_21d,min_63d,min_126d,max_21d,max_63d,max_126d,exp_mean_21d,exp_mean_63d,exp_mean_126d,target
0,-0.1,0.2,0.2,0.3,0.4,0.4,5.0,4.7,4.1,-0.2,...,4.6,3.9,3.3,5.2,5.2,5.2,4.9,4.64,4.23,-0.040896
0,0.0,0.3,0.2,0.4,0.4,0.4,5.0,4.6,4.1,0.1,...,4.6,3.7,3.3,5.2,5.2,5.2,5.0,4.63,4.19,-0.061932


In [15]:
#query_instance = X_test.iloc[0].values
counterfactual = cfs.drop(columns="target")[:1].values.astype(np.float32)


In [245]:
session.run(None, {input_name: counterfactual})[0]

array([[-0.02516202]], dtype=float32)

## Generate counterfactuals

In [17]:
print(training_data["target"].min(),training_data["target"].max())

-0.9272918124562632 11.238508633824596


In [18]:
print(testing_data["target"].min(),testing_data["target"].max())

-0.9829017264276227 4.541095890410959


In [469]:
all_cf = []

n = len(testing_data.iloc[:10,:])

min, max = training_data["target"].min(), training_data["target"].max()

for i in range(n):
    query_instance = testing_data.drop(columns="target").iloc[i:i+1].astype(np.float64)
    factual_id = query_instance.index[0]

    dice_exp = exp.generate_counterfactuals(
        query_instance,
        total_CFs=1,
        desired_range=[min, max]
    )
    
    cf_df = dice_exp.cf_examples_list[0].final_cfs_df.copy()
    cf_df["factual_id"] = factual_id  # rattache à son factuel

    all_cf.append(cf_df)

# Concaténer tous les contre-factuels
all_cf_df = pd.concat(all_cf, ignore_index=True)

100%|██████████| 1/1 [00:00<00:00,  1.50it/s]


In [470]:
all_cf_df

,past_profitability_21d,past_profitability_63d,past_profitability_126d,volatility_21d,volatility_63d,volatility_126d,avg_price_21d,avg_price_63d,avg_price_126d,sharpe_21d,...,min_63d,min_126d,max_21d,max_63d,max_126d,exp_mean_21d,exp_mean_63d,exp_mean_126d,target,factual_id
0,0.0,0.3,0.2,0.4,0.4,0.4,5.0,4.6,4.1,0.1,...,3.7,3.3,5.2,5.2,5.2,5.0,4.63,4.19,-0.061932,0
1,0.0,0.3,0.2,0.4,0.4,0.4,5.0,4.6,4.1,0.1,...,3.7,3.3,5.2,5.2,5.2,5.0,4.63,4.19,-0.061932,1
2,0.0,0.3,0.2,0.4,0.4,0.4,5.0,4.6,4.1,0.1,...,3.7,3.3,5.2,5.2,5.2,5.0,4.63,4.19,-0.061932,2
3,0.0,0.4,0.2,0.4,0.4,0.4,5.0,4.6,4.1,0.1,...,3.6,3.3,5.2,5.2,5.2,5.0,4.63,4.18,-0.072740,3
4,-0.1,0.1,0.2,0.3,0.4,0.3,4.7,4.4,4.1,-0.2,...,4.0,3.6,4.9,4.9,4.9,4.7,4.45,4.19,0.116457,4
5,-0.1,0.1,0.3,0.3,0.4,0.4,5.2,4.9,4.6,-0.2,...,4.3,3.7,5.5,5.6,5.6,5.1,4.93,4.64,-0.192252,5
6,-0.1,0.1,0.3,0.3,0.4,0.4,5.2,4.9,4.6,-0.2,...,4.3,3.7,5.5,5.6,5.6,5.1,4.93,4.64,-0.192252,6
7,-0.1,0.1,0.3,0.3,0.4,0.4,5.2,4.9,4.6,-0.2,...,4.3,3.7,5.5,5.6,5.6,5.1,4.93,4.64,-0.192252,7
8,-0.1,0.1,0.3,0.4,0.4,0.4,5.2,4.9,4.6,-0.2,...,4.3,3.7,5.5,5.6,5.6,5.1,4.92,4.64,-0.191086,8
9,-0.1,0.1,0.3,0.4,0.4,0.4,5.2,4.9,4.6,-0.2,...,4.3,3.7,5.5,5.6,5.6,5.1,4.92,4.64,-0.191086,9


In [ ]:
# Sauvegarder en CSV
# all_cf_df.to_csv("counterfactuals_500.csv", index=False)
# print("Saved counterfactuals shape:", all_cf_df.shape)

Saved counterfactuals shape: (500, 32)


In [21]:
from termcolor import colored

def highlight_changes(factual, counterfactual):
    """Affiche chaque feature, en rouge si modifiée"""
    for col in factual.index:
        f_val = factual[col]
        c_val = counterfactual[col]
        if f_val != c_val:
            print(col, ":", colored(f_val, "green"), "→", colored(c_val, "red"))
        else:
            print(col, ":", f_val)
    print("-" * 40)


In [388]:
# Exemple : visualiser pour le premier factuel
factual_id = all_cf_df["factual_id"].iloc[0]
factual = testing_data.loc[factual_id]

subset = all_cf_df[all_cf_df["factual_id"] == factual_id]

for idx, row in subset.iterrows():
    print(f"\nCounterfactual {idx}:")
    highlight_changes(factual, row.drop("factual_id"))


Counterfactual 0:
past_profitability_21d : -0.0738290311921911 → 0.0
past_profitability_63d : 0.213770214095279 → 0.3
past_profitability_126d : 0.2583447661125139 → 0.2
volatility_21d : 0.3102627523423307 → 0.4
volatility_63d : 0.4338144270914076 → 0.4
volatility_126d : 0.4058433499915637 → 0.4
avg_price_21d : 4.959744761904762 → 5.0
avg_price_63d : 4.7161784126984125 → 4.6
avg_price_126d : 4.147089206349207 → 4.1
sharpe_21d : -0.236372434043557 → 0.1
sharpe_63d : 0.4937089261175924 → 0.8
sharpe_126d : 0.6372400189046246 → 0.6
m_21d : -0.3750000000000001 → 0.2
m_63d : 0.825 → 1.2
m_126d : 0.9599999999999996 → 0.9
roc_21d : -0.0738290311921911 → 0.0
roc_63d : 0.213770214095279 → 0.3
roc_126d : 0.2583447661125139 → 0.2
MACD : -0.033727719892717 → 0.0
rsi_14 : 36.63514971770492 → 36.8
dco_22 : 0.1235163636363635 → 0.1
min_21d : 4.614 → 4.6
min_63d : 3.9129999999999994 → 3.7
min_126d : 3.32 → 3.3
max_21d : 5.19 → 5.2
max_63d : 5.19 → 5.2
max_126d : 5.19 → 5.2
exp_mean_21d : 4.868377232648

## Evaluation

In [532]:
from typing import List, Tuple, Union

def l0_distance(delta: np.ndarray) -> List[float]:
    """
    Computes L-0 norm, number of non-zero entries.

    Parameters
    ----------
    delta: np.ndarray
        Difference between factual and counterfactual

    Returns
    -------
    List[float]
    """
    # get mask that selects all elements that are NOT zero (with some small tolerance)
    difference_mask = np.invert(np.isclose(delta, np.zeros_like(delta), atol=1e-05))
    # get the number of changed features for each row
    num_feature_changes = np.sum(
        difference_mask,
        axis=1,
        dtype=float,
    )
    distance = num_feature_changes.tolist()
    return distance

In [533]:
def l1_distance(delta: np.ndarray) -> List[float]:
    """
    Computes L-1 distance, sum of absolute difference.

    Parameters
    ----------
    delta: np.ndarray
        Difference between factual and counterfactual

    Returns
    -------
    List[float]
    """
    absolute_difference = np.abs(delta)
    distance = np.sum(absolute_difference, axis=1, dtype=float).tolist()
    return distance

In [534]:
def l2_distance(delta: np.ndarray) -> List[float]:
    """
    Computes L-2 distance, sum of squared difference - Euclidean distance.

    Parameters
    ----------
    delta: np.ndarray
        Difference between factual and counterfactual

    Returns
    -------
    List[float]
    """
    squared_difference = np.square(np.abs(delta))
    distance = np.sum(squared_difference, axis=1, dtype=float).tolist()
    return distance


In [535]:
def manhattan_distance(numeric_cols:list, mad_values: dict, factual: np.ndarray, counterfactual: np.ndarray) -> List[float]:
    """
    Computes Manhattan distance.

    Parameters
    ----------
    delta: np.ndarray
        Difference between factual and counterfactual

    Returns
    -------
    List[float]
    """
    # Compute distance pairwise
    distance = 0
    keys_list = list(mad_values.keys())

    for idx in numeric_cols[:-1]: # make sure the target is not included in the distance calculation
        col_name = keys_list[idx]
        distance += abs(factual[idx] - counterfactual[idx]) / mad_values[col_name]

    return [distance]

In [536]:
def _get_delta(factual: np.ndarray, counterfactual: np.ndarray) -> np.ndarray:
    """
    Compute difference between original factual and counterfactual

    Parameters
    ----------
    factual: np.ndarray
        Normalized and encoded array with factual data.
        Shape: NxM
    counterfactual: : np.ndarray
        Normalized and encoded array with counterfactual data.
        Shape: NxM

    Returns
    -------
    np.ndarray
    """
    return counterfactual - factual

In [537]:
def _get_distances(factual: np.ndarray, counterfactual: np.ndarray) -> List[List[float]]:
    """
    Computes distances.
    All features have to be in the same order (without target label).

    Parameters
    ----------
    factual: np.ndarray
        Normalized and encoded array with factual data.
        Shape: NxM
    counterfactual: np.ndarray
        Normalized and encoded array with counterfactual data
        Shape: NxM

    Returns
    -------
    list: distances 1 to 4
    """
    if factual.shape != counterfactual.shape:
        raise ValueError("Shapes of factual and counterfactual have to be the same")
    if len(factual.shape) != 2:
        raise ValueError(
            "Shapes of factual and counterfactual have to be 2-dimensional"
        )

    # get difference between original and counterfactual
    delta = _get_delta(factual, counterfactual)

    d0 = l0_distance(delta)
    d1 = l1_distance(delta)
    d2 = l2_distance(delta)

    return [[d0[i],d1[i],d2[i]] for i in range(len(d2))]

In [538]:
from abc import ABC, abstractmethod

import pandas as pd


class Evaluation(ABC):
    def __init__(self, mlmodel, hyperparameters: dict = None):
        """

        Parameters
        ----------
        mlmodel:
            Classification model. (optional)
        hyperparameters:
            Dictionary with hyperparameters, could be used to pass other things. (optional)
        """
        self.mlmodel = mlmodel
        self.hyperparameters = hyperparameters

    @abstractmethod
    def get_evaluations(
        self, factuals: pd.DataFrame, counterfactuals: pd.DataFrame
    ) -> pd.DataFrame:
        """Compute evaluation measure"""
    
    @abstractmethod
    def get_manhattan(
        self, factuals: pd.DataFrame, counterfactuals: pd.DataFrame
    ) -> pd.DataFrame:
        """Compute evaluation measure"""

In [539]:
def remove_nans(
    counterfactuals: pd.DataFrame, factuals: pd.DataFrame = None
) -> Union[Tuple[pd.DataFrame, pd.DataFrame], pd.DataFrame]:
    """Remove instances for which a counterfactual could not be found.

    Parameters
    ----------
    counterfactuals:
        Has to be the same shape as factuals.
    factuals:
        Has to be the same shape as counterfactuals. (optional)

    Returns
    -------

    """
    # get indices of unsuccessful counterfactuals
    nan_idx = counterfactuals.index[counterfactuals.isnull().any(axis=1)]
    output_counterfactuals = counterfactuals.copy()
    output_counterfactuals = output_counterfactuals.drop(index=nan_idx)

    if factuals is not None:
        if factuals.shape[0] != counterfactuals.shape[0]:
            raise ValueError(
                "Counterfactuals and factuals should contain the same amount of samples"
            )
        output_factuals = factuals.copy()
        output_factuals = output_factuals.drop(index=nan_idx)
        return output_counterfactuals, output_factuals

    return output_counterfactuals

In [540]:
class Distance(Evaluation):
    """
    Calculates the L0, L1, L2, and L-infty distance measures.
    """

    def __init__(self, mlmodel):
        super().__init__(mlmodel)
        # self.columns = ["L0_distance", "L1_distance", "L2_distance", "Linf_distance"]
        self.columns = ["L0_distance", "L1_distance", "L2_distance"]
    def get_evaluations(self, factuals, counterfactuals):
        # only keep the rows for which counterfactuals could be found
        counterfactuals_without_nans, factuals_without_nans = remove_nans(
            counterfactuals, factuals
        )

        # return empty dataframe if no successful counterfactuals
        if counterfactuals_without_nans.empty:
            return pd.DataFrame(columns=self.columns)

        distances = []
        for idx in factuals.index:
            arr_f = factuals_without_nans.iloc[idx].to_numpy(dtype=np.float64)
            arr_cf = counterfactuals_without_nans.iloc[idx].to_numpy(dtype=np.float64)
            distances.append(_get_distances(arr_f.reshape(1, -1), arr_cf.reshape(1, -1)))

        # Flatten outer list
        flat_distances = [row[0] for row in distances]

        return pd.DataFrame(flat_distances, columns=self.columns)
    
    def get_manhattan(self, factuals, counterfactuals):
        mad_values = exp.data_interface.get_valid_mads()
        # only keep the rows for which counterfactuals could be found
        counterfactuals_without_nans, factuals_without_nans = remove_nans(
            counterfactuals, factuals
        )

        # return empty dataframe if no successful counterfactuals
        if counterfactuals_without_nans.empty:
            return pd.DataFrame(columns=["Manhattan Distance"])

        numerical_cols = factuals_without_nans.select_dtypes(include=["number"]).columns.tolist()
        numerical_cols_idx = [factuals_without_nans.columns.get_loc(c) for c in numerical_cols]

        distances = []
        for idx in factuals.index:
            arr_f = factuals_without_nans.iloc[idx].to_numpy(dtype=np.float64)
            arr_cf = counterfactuals_without_nans.iloc[idx].to_numpy(dtype=np.float64)
            distances.append(manhattan_distance(numerical_cols_idx,mad_values,arr_f, arr_cf))

        return pd.DataFrame(distances, columns=["Manhattan Distance"])
    

In [513]:
factuals = testing_data[testing_data.index.isin(all_cf_df['factual_id'])]

In [471]:
counterfactuals = all_cf_df.copy()

In [541]:
# Initialize distance evaluator
dist_eval = Distance(dice_model)


In [474]:
manhattan_dist = dist_eval.get_manhattan(factuals, counterfactuals)
manhattan_dist

,Manhattan Distance
0,18.510589
1,21.079296
2,23.561178
3,29.907345
4,8.940789
5,6.039410
6,7.884227
7,8.666770
8,9.670428
9,9.065631


### Standardize factuals and counterfactuals for other distance metrics

In [490]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(training_data.drop(columns="target"))

all_cf_scaled = []
all_factuals_scaled = []

n = len(testing_data.iloc[:10,:])

for i in range(n):
    query_instance = testing_data.drop(columns="target").iloc[i:i+1].astype(np.float64)
    factual_id = query_instance.index[0]
    # Standardize factual
    factual_scaled = pd.DataFrame(
        scaler.transform(query_instance),
        columns=query_instance.columns,
        index=query_instance.index
    )
    all_factuals_scaled.append(factual_scaled)

    dice_exp = exp.generate_counterfactuals(
        query_instance,
        total_CFs=1,
        desired_range=[min, max]
    )
    #cf_df = dice_exp.cf_examples_list[0].final_cfs_df.copy()
    cfs = dice_exp.cf_examples_list[0].final_cfs_df.copy()
    query_instance["target"] = testing_data["target"].iloc[i:i+1].values.astype(np.float64)

    # Standardize counterfactual
    counterfactual_scaled = pd.DataFrame(
        scaler.transform(cfs.drop(columns="target")),
        columns=cfs.columns[:-1],
        index=cfs.index
    )
    all_cf_scaled.append(counterfactual_scaled)

# Concatenate all single-row DataFrames into one DataFrame each
all_factuals_scaled_df = pd.concat(all_factuals_scaled, axis=0)
all_cf_scaled_df = pd.concat(all_cf_scaled, axis=0)

100%|██████████| 1/1 [00:00<00:00,  1.25it/s]


In [493]:
all_factuals_scaled_df.shape, all_cf_scaled_df.shape

((10, 30), (10, 30))

In [542]:
all_distances = dist_eval.get_evaluations(all_factuals_scaled_df, all_cf_scaled_df)
all_distances

,L0_distance,L1_distance,L2_distance
0,30.0,4.468350,2.581912
1,30.0,4.999619,3.051661
2,30.0,5.505026,3.758377
3,30.0,6.813707,5.955110
4,30.0,1.779387,0.291525
5,30.0,1.320388,0.206285
6,30.0,1.543202,0.263952
7,30.0,1.695073,0.286288
8,30.0,1.948371,0.376131
9,30.0,1.850676,0.364313


## CARLA

For classification problem! It also has evaluation metrics

In [ ]:
from carla.data.catalog import CsvCatalog

#continuous = ["age", "fnlwgt", "education-num", "capital-gain", "hours-per-week", "capital-loss"]
#categorical = ["marital-status", "native-country", "occupation", "race", "relationship", "sex", "workclass"]
#immutable = ["age", "sex"]
file_path = "training_data.csv" # Path to your data

dataset = CsvCatalog(file_path=file_path,
                    continuous=continuous,
                    categorical=[],
                    immutables=[],
                    target='target')

display(dataset.df.head())

,past_profitability_21d,past_profitability_63d,past_profitability_126d,volatility_21d,volatility_63d,...,max_126d,exp_mean_21d,exp_mean_63d,exp_mean_126d,target
0,0.136332,0.169568,0.211132,0.008955,0.017334,...,0.0011,0.000881,0.000898,0.000927,0.048596
1,0.135998,0.173294,0.212602,0.009270,0.017363,...,0.0011,0.000884,0.000898,0.000927,0.055726
2,0.133644,0.172705,0.210647,0.009641,0.017483,...,0.0011,0.000885,0.000898,0.000926,0.068956
3,0.130236,0.171356,0.204992,0.010008,0.017608,...,0.0011,0.000885,0.000898,0.000925,0.087021
4,0.127791,0.168613,0.199268,0.010343,0.017683,...,0.0011,0.000884,0.000897,0.000924,0.116883


In [3]:
# load catalog model
model_type = "forest"
ml_model = MLModelCatalog(
    dataset,
    model_type=model_type,
    load_online=True,
    backend="sklearn"
)

In [4]:


# define your recourse method
recourse_method = recourse_catalog.Dice(ml_model, hyperparams={})


In [5]:
# get some negative instances
factuals = predict_negative_instances(ml_model, dataset.df)
factuals = factuals[:5]

# find counterfactuals
counterfactuals = recourse_method.get_counterfactuals(factuals)

In [6]:
counterfactuals

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
0,0.301370,0.044131,0.800000,0.02174,0.0,0.459799
1,0.452055,0.048052,0.800000,0.00000,0.0,0.946284
2,0.287671,0.137581,0.858199,0.00000,0.0,0.936285
3,0.493151,0.150486,0.952120,0.00000,0.0,0.823724
4,0.150685,0.220635,0.800000,0.70000,0.0,0.397959


In [7]:
factuals

,age,fnlwgt,education-num,capital-gain,capital-loss,...,occupation_Other,race_White,relationship_Non-Husband,sex_Male,workclass_Private
0,0.301370,0.044131,0.800000,0.02174,0.0,...,0.0,1.0,1.0,1.0,0.0
1,0.452055,0.048052,0.800000,0.00000,0.0,...,0.0,1.0,0.0,1.0,0.0
2,0.287671,0.137581,0.533333,0.00000,0.0,...,1.0,1.0,1.0,1.0,1.0
3,0.493151,0.150486,0.400000,0.00000,0.0,...,1.0,0.0,0.0,1.0,1.0
4,0.150685,0.220635,0.800000,0.00000,0.0,...,0.0,0.0,1.0,0.0,1.0
